# Exercise 2: Metric comparison

*Machine Translation · version 0.1.0*

> Take 20 English sentences and translate them into Turkish using `facebook/nllb-200-distilled-600M`, with two decoding settings (for example greedy and beam search) as competing systems. Use references written or checked by a competent speaker, such as the human-post-edited X-WMT test sets. Compute corpus BLEU and chrF++, retaining metric signatures. For sentence-level diagnostics, state the BLEU smoothing and effective-order settings. Inspect examples where the two metrics rank competing outputs differently; their raw score magnitudes are not directly comparable. Distinguish legitimate variants from meaning-changing suffix errors.

**Setup.** `pip install "turkicnlp[hfst]==0.5.0" torch transformers sentencepiece sacrebleu requests` (the compiled `hfst` package needs Python 3.11 or 3.12; elsewhere install `pyhfst`). NLLB-200 distilled 600M is loaded from `NLLB_PATH` (a local copy) or from the Hugging Face Hub at the revision pinned in the book; it runs in float32 on the CPU, about 1.7 s per sentence on four cores. Translations are cached in `BOOK_DATA_DIR` (default `~/.cache/turkic-nlp-book`), so an interrupted run resumes where it stopped. Test data come from the commit-pinned TIL release (X-WMT test sets, MIT licence); the X-WMT files are tokenised, and the code removes the spaces around punctuation before scoring (a heuristic, see `detok`). Scores use SacreBLEU with its default settings: BLEU with the `13a` tokeniser, chrF, and chrF++ (`word_order=2`).

The 20 sentences are the first 20 of the X-WMT English–Turkish test set. Sentence-level BLEU uses `effective_order=True`, so short sentences without higher-order matches are not scored as zero.

In [ ]:
import collections, hashlib, os, re, statistics, time
from pathlib import Path
import requests

# Commit-pinned TIL release files (X-WMT test sets and the corpus README); MIT licence.
TIL = "https://raw.githubusercontent.com/turkic-interlingua/til-mt/7d988cc47dba889d02262d893bf9a46e43a6bb36/"
CACHE = Path(os.environ.get("BOOK_DATA_DIR", "~/.cache/turkic-nlp-book")).expanduser()
FETCHED = {}                                   # url -> sha256 of every file read (saved with the outputs)

def fetch(url):
    path = CACHE / hashlib.sha1(url.encode()).hexdigest()[:12] / url.rsplit("/", 1)[-1]
    if not path.exists():
        r = requests.get(url, timeout=300)
        r.raise_for_status()
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(r.content)
    FETCHED[url] = hashlib.sha256(path.read_bytes()).hexdigest()
    return path

def detok(s):
    """Undo the X-WMT tokenisation (spaces around punctuation) so references match raw MT output."""
    s = re.sub(r" ([’'] ?)", lambda m: m.group(1).strip(), s)       # Sialkot ’ taki -> Sialkot’taki
    s = re.sub(r" ([.,!?;:%)\]»])", r"\1", s)
    return re.sub(r"([(\[«]) ", r"\1", s).strip()

def xwmt(src, tgt, n=None):
    """The first n sentence pairs of the X-WMT test set for src-tgt (X-WMT language codes)."""
    s = fetch(f"{TIL}xwmt/test/{src}-{tgt}/{src}-{tgt}.{src}").read_text(encoding="utf-8").splitlines()
    t = fetch(f"{TIL}xwmt/test/{src}-{tgt}/{src}-{tgt}.{tgt}").read_text(encoding="utf-8").splitlines()
    assert len(s) == len(t), (src, tgt)
    pairs = [(detok(a), detok(b)) for a, b in zip(s, t) if a.strip() and b.strip()]
    return pairs[:n] if n else pairs

NLLB_CODE = {"az": "azj_Latn", "ba": "bak_Cyrl", "kk": "kaz_Cyrl", "ky": "kir_Cyrl", "tr": "tur_Latn",
             "uz": "uzn_Latn", "en": "eng_Latn", "ru": "rus_Cyrl"}
NLLB = {}

def nllb():
    """NLLB-200 distilled 600M: from NLLB_PATH (a local copy) or the pinned Hub revision; float32, CPU."""
    if not NLLB:
        import torch
        from transformers import AutoModelForSeq2SeqLM, NllbTokenizer
        path = os.environ.get("NLLB_PATH", "facebook/nllb-200-distilled-600M")
        rev = None if Path(path).exists() else "f8d333a098d19b4fd9a8b18f94170487ad3f821d"
        NLLB["tok"] = NllbTokenizer.from_pretrained(path, revision=rev)
        NLLB["model"] = AutoModelForSeq2SeqLM.from_pretrained(path, revision=rev, torch_dtype=torch.float32).eval()
        NLLB["torch"] = torch
        torch.set_num_threads(os.cpu_count() or 1)
    return NLLB["tok"], NLLB["model"]

CACHE_FILE = CACHE / "nllb-translations.jsonl"       # translations are cached, so a run can resume
LAST_SECONDS = 0.0

def _cached():
    import json
    out = {}
    if CACHE_FILE.exists():
        for line in CACHE_FILE.read_text(encoding="utf-8").splitlines():
            r = json.loads(line); out[tuple(r["key"])] = (r["out"], r["capped"], r["sec"])
    return out

def translate(sents, src, tgt, beams=1, batch=8, max_new=256):
    """Translate with explicit source and target language tokens (greedy unless beams > 1).
    Returns (outputs, number of outputs that reached the generation ceiling without an
    end-of-sequence token). Results are cached per sentence in CACHE_FILE, with the generation
    time of their batch divided by its size; LAST_SECONDS holds the total for the last call."""
    global LAST_SECONDS
    import json
    cache = _cached()
    key = lambda s: [src, tgt, beams, max_new, s]
    todo = [s for s in dict.fromkeys(sents) if tuple(key(s)) not in cache]
    if todo:
        tok, model = nllb()
        tok.src_lang = NLLB_CODE[src]
        CACHE_FILE.parent.mkdir(parents=True, exist_ok=True)
        with NLLB["torch"].inference_mode(), CACHE_FILE.open("a", encoding="utf-8") as f:
            for i in range(0, len(todo), batch):
                chunk = todo[i:i + batch]
                t0 = time.time()
                enc = tok(chunk, return_tensors="pt", padding=True)
                gen = model.generate(**enc, forced_bos_token_id=tok.convert_tokens_to_ids(NLLB_CODE[tgt]),
                                     num_beams=beams, do_sample=False, max_new_tokens=max_new)
                sec = (time.time() - t0) / len(chunk)
                for s, row, text in zip(chunk, gen, tok.batch_decode(gen, skip_special_tokens=True)):
                    capped = int(row[-1] != tok.eos_token_id and row[-1] != tok.pad_token_id)
                    cache[tuple(key(s))] = (text, capped, sec)
                    f.write(json.dumps({"key": key(s), "out": text, "capped": capped, "sec": sec}, ensure_ascii=False) + "\n")
                f.flush()
    res = [cache[tuple(key(s))] for s in sents]
    LAST_SECONDS = sum(r[2] for r in res)
    return [r[0] for r in res], sum(r[1] for r in res)

def metrics(hyps, refs):
    """Corpus BLEU (13a), chrF and chrF++ with SacreBLEU, rounded, plus the chrF++ signature."""
    from sacrebleu.metrics import BLEU, CHRF
    bleu, chrf, chrfpp = BLEU(tokenize="13a"), CHRF(), CHRF(word_order=2)
    r = {"BLEU": bleu.corpus_score(hyps, [refs]).score, "chrF": chrf.corpus_score(hyps, [refs]).score,
         "chrF++": chrfpp.corpus_score(hyps, [refs]).score}
    return {k: round(v, 2) for k, v in r.items()}, str(chrfpp.get_signature())

In [ ]:
from sacrebleu.metrics import BLEU, CHRF
pairs = xwmt("en", "tr", 20)                    # professional X-WMT references (English -> Turkish)
src, refs = [s for s, _ in pairs], [t for _, t in pairs]
systems = {}
for name, beams in [("greedy", 1), ("beam 4", 4)]:
    systems[name], capped = translate(src, "en", "tr", beams=beams)
    scores, sig = metrics(systems[name], refs)
    print(f"{name:7} {scores}  ({LAST_SECONDS:.0f} s generation time, {capped} at the length ceiling)")
print("chrF++ signature:", sig)

# Sentence level: where do BLEU and chrF++ prefer different systems?
sbleu, schrf = BLEU(tokenize="13a", effective_order=True), CHRF(word_order=2)
disagree = []
for i, ref in enumerate(refs):
    a, b = systems["greedy"][i], systems["beam 4"][i]
    if a == b:
        continue
    db = sbleu.sentence_score(b, [ref]).score - sbleu.sentence_score(a, [ref]).score
    dc = schrf.sentence_score(b, [ref]).score - schrf.sentence_score(a, [ref]).score
    if db * dc < 0:
        disagree.append((i, db, dc))
print(f"{sum(a != b for a, b in zip(systems['greedy'], systems['beam 4']))} of 20 outputs differ; "
      f"BLEU and chrF++ disagree on which is better for {len(disagree)}")
for i, db, dc in disagree[:3]:
    print(f"\n#{i}  BLEU(beam-greedy) {db:+.1f}  chrF++(beam-greedy) {dc:+.1f}")
    print("  ref   :", refs[i]); print("  greedy:", systems["greedy"][i]); print("  beam 4:", systems["beam 4"][i])

## Worked solution

Output of `experiments/ch10_exercises/run_ch10_exercises.py` (NLLB-200 distilled 600M, CPU, float32):

```
greedy  {'BLEU': 9.78, 'chrF': 48.8, 'chrF++': 43.5}  (36 s generation time, 0 at the length ceiling)
beam 4  {'BLEU': 10.23, 'chrF': 50.51, 'chrF++': 45.21}  (95 s generation time, 0 at the length ceiling)
chrF++ signature: nrefs:1|case:mixed|eff:yes|nc:6|nw:2|space:no|version:2.6.0
20 of 20 outputs differ; BLEU and chrF++ disagree on which is better for 9

#1  BLEU(beam-greedy) +0.2  chrF++(beam-greedy) -0.4
  ref   : Yetkililer mevcut en son bilginin Pakistan bazlı terör örgütünün, su altı saldırılar için örgüt üyelerini eğittiğini söyledi.
  greedy: Yetkililer, Pakistan'da bulunan terörist kamplarının sualtı saldırılar için kadroları eğitmeye devam ettiğini söylüyor.
  beam 4: Yetkililer, Pakistan merkezli terörist kamplarının sualtı saldırılar için kadroları eğitmeye devam ettiklerini söylüyorlar.

#2  BLEU(beam-greedy) -0.1  chrF++(beam-greedy) +0.5
  ref   : Balakot kampı Jaish-e-Mohammad tarafından yeniden aktif hale gelmişken Lashkar-e-Tayyiba da Mirpur ve Sialkot’taki iki tane deniz kampını yeniden aktif hale getirdi.
  greedy: Balakot kampı Jaish-e-Mohammed tarafından yeniden etkinleştirildiği sürece, Lashkar-e-Tayiba Mirpur ve Sialkot'ta iki deniz kampını etkinleştirdi.
  beam 4: Balakot kampı Jaish-e-Mohammed tarafından yeniden etkinleştirildiğinde, Lashkar-e-Tayiba Mirpur ve Sialkot'ta iki deniz kampını etkinleştirdi.

#4  BLEU(beam-greedy) +2.6  chrF++(beam-greedy) -7.9
  ref   : Yetkililerin OneIndia’ya dediğine göre, terör grupları devasa güvenlik güçlendirmeleri karşısında karaya sahip olmak zorlaştığı için çeşitli yollara başvuracaktır.
  greedy: Yetkililer OneIndia'ya, büyük güvenlik güçleri nedeniyle topraklara erişimi zorlaştığı için terör gruplarının çeşitli teknikleri kullanmaya çalışacağını söylüyor.
  beam 4: Yetkililer OneIndia'ya terör örgütlerinin toprağa erişilebilmesi zorlaştığı için çeşitli teknikler kullanmaya çalışacağını söylüyorlar.
```

**Corpus scores.** Beam search (4 beams) is slightly better than greedy decoding on both metrics: BLEU 10.23 against 9.78, chrF++ 45.21 against 43.50, at about 2.6 times the generation time. On 20 sentences such differences are within the noise; they are a demonstration, not a ranking.

**Where the metrics disagree.** All 20 outputs differ between the two systems, and for 9 of them sentence-level BLEU and chrF++ prefer different systems. Three are printed:

- In #2 greedy decoding writes *yeniden etkinleştirildiği sürece* 'as long as it was reactivated' and beam search *etkinleştirildiğinde* 'when it was reactivated'. The reference has *aktif hale gelmişken* 'while it had become active'. The greedy suffix *-DIğI sürece* changes the meaning to a condition; the beam output keeps the temporal reading. chrF++ prefers the beam output, BLEU (barely) the greedy one.
- In #1 the outputs differ in agreement and tense (*devam ettiğini söylüyor* vs. *devam ettiklerini söylüyorlar*). Both are acceptable Turkish; plural agreement with an organisation subject is a legitimate variant, and neither matches the reference's past tense *söyledi*.
- In #4 the beam output drops the phrase about security forces (*büyük güvenlik güçleri nedeniyle*), a real adequacy loss. chrF++ penalises it by 7.9 points, while BLEU rises by 2.6, because the shorter output happens to match more of the reference's word n-grams.

The small sentence-level differences in #1 and #2 are within what either metric can resolve; the meaning difference in #2 is only visible to a reader. Both apostrophe spellings (*Sialkot'ta* vs. the reference *Sialkot’taki*) also cost points without any translation error.

## Your turn — extension

1. Add a third system (for example the Kazakh–Turkish Tilmash model or an LLM) and compute COMET with a recorded checkpoint.
2. Normalise apostrophes (’ → ') in hypotheses and references and measure how much each metric changes.